In [80]:
from google.colab import files
uploaded = files.upload()

Saving shuttle.csv to shuttle.csv


In [ ]:
%%writefile decision_tree.cpp
// ===================================================
// MILESTONE 1 — Sequential CPU Decision Tree Builder
// ===================================================

#include <algorithm>
#include <cassert>
#include <cmath>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <limits>
#include <map>
#include <numeric>
#include <random>
#include <sstream>
#include <stdexcept>
#include <string>
#include <vector>
#include <chrono>

// ── Timing accumulators (global) ─────────────────────────────────────────
double g_split_eval_time_sec = 0.0;

// =======================================
// ── FARAZ'S DATA STRUCTURES & UTILITIES
// =======================================

struct Dataset {
    std::vector<std::vector<float>> features;
    std::vector<int>                labels;
    std::vector<std::string>        feature_names;
};

// Node structure — Faraz defined this; Yaman FILLS and USES it.
struct Node {
    std::vector<int> sample_indices;   // which training samples live here
    int              depth;
    int              feature_index;    // split feature (-1 if leaf)
    float            threshold;        // split threshold
    Node*            left;
    Node*            right;
    bool             is_leaf;
    int              predicted_class;  // majority class at this node

    Node(int d = 0)
        : depth(d), feature_index(-1), threshold(0.0f),
          left(nullptr), right(nullptr),
          is_leaf(false), predicted_class(-1) {}
};

// ── Faraz: load CSV (last column = label) ───────────────────────────────────
Dataset load_csv(const std::string& filename, bool has_header = true) {
    Dataset data;
    std::ifstream file(filename);
    if (!file.is_open()) {
        std::cerr << "[!] Cannot open: " << filename << "\n";
        return data;
    }
    std::string line;
    int line_number = 0;
    while (std::getline(file, line)) {
        ++line_number;
        if (line.empty()) continue;
        if (has_header && line_number == 1) {
            std::stringstream ss(line);
            std::string item;
            while (std::getline(ss, item, ','))
                data.feature_names.push_back(item);
            if (!data.feature_names.empty())
                data.feature_names.pop_back();
            continue;
        }
        std::stringstream ss(line);
        std::string value;
        std::vector<float> row;
        float label_val = 0;
        while (std::getline(ss, value, ',')) {
            float num = std::stof(value);
            if (ss.peek() == EOF) label_val = num;
            else                  row.push_back(num);
        }
        data.features.push_back(row);
        data.labels.push_back(static_cast<int>(label_val));
    }
    return data;
}

// ── Faraz: train/test split by indices ──────────────────────────────────────
std::pair<std::vector<int>, std::vector<int>>
train_test_split(const Dataset& data, float train_ratio = 0.8f, int seed = 42) {
    int n = static_cast<int>(data.features.size());
    std::vector<int> idx(n);
    std::iota(idx.begin(), idx.end(), 0);
    std::shuffle(idx.begin(), idx.end(), std::default_random_engine(seed));
    int train_size = static_cast<int>(n * train_ratio);
    return { std::vector<int>(idx.begin(), idx.begin() + train_size),
             std::vector<int>(idx.begin() + train_size, idx.end()) };
}

// ── Faraz: create root node ──────────────────────────────────────────────────
Node* create_root_node(const std::vector<int>& train_indices) {
    Node* root = new Node(0);
    root->sample_indices = train_indices;
    return root;
}

// ── Faraz: get samples at a node ────────────────────────────────────────────
const std::vector<int>& get_samples_for_node(Node* node) {
    return node->sample_indices;
}

// ── Faraz: split a node into two children ───────────────────────────────────
std::pair<Node*, Node*> split_node(Node* parent,
                                   const std::vector<int>& left_indices,
                                   const std::vector<int>& right_indices) {
    Node* left  = new Node(parent->depth + 1);
    Node* right = new Node(parent->depth + 1);
    left->sample_indices  = left_indices;
    right->sample_indices = right_indices;
    parent->left  = left;
    parent->right = right;
    return {left, right};
}

// =========================
// ── FATIMA'S SPLIT-FINDER
// =========================

struct SplitResult {
    int    best_feature   = -1;
    int    best_bin_index = -1;
    double best_threshold = 0.0;
    double best_gini      = std::numeric_limits<double>::infinity();
    bool   found          = false;
};

static double gini_from_counts(const std::vector<int>& counts) {
    int total = 0;
    for (int c : counts) total += c;
    if (total == 0) return 0.0;
    double sum_sq = 0.0;
    for (int c : counts) {
        double p = static_cast<double>(c) / total;
        sum_sq += p * p;
    }
    return 1.0 - sum_sq;
}

static int find_num_classes(const std::vector<int>& y) {
    int mx = -1;
    for (int v : y) {
        if (v < 0) throw std::invalid_argument("Negative label.");
        if (v > mx) mx = v;
    }
    return mx + 1;
}

static std::vector<double> equal_width_boundaries(const std::vector<double>& vals, int bins) {
    auto [mn_it, mx_it] = std::minmax_element(vals.begin(), vals.end());
    double mn = *mn_it, mx = *mx_it;
    std::vector<double> b(bins + 1);
    if (mn == mx) { for (auto& x : b) x = mn; return b; }
    double w = (mx - mn) / bins;
    for (int i = 0; i <= bins; ++i) b[i] = mn + i * w;
    b.back() = mx;
    return b;
}

static int val_to_bin(double x, const std::vector<double>& b) {
    int bc = static_cast<int>(b.size()) - 1;
    if (x <= b.front()) return 0;
    if (x >= b.back())  return bc - 1;
    auto it = std::upper_bound(b.begin(), b.end(), x);
    int idx = static_cast<int>(it - b.begin()) - 1;
    return std::clamp(idx, 0, bc - 1);
}

SplitResult find_best_histogram_split(
    const std::vector<std::vector<double>>& X,
    const std::vector<int>& y,
    int bin_count)
{
    if (X.empty() || X.size() != y.size() || bin_count < 2) return {};
    int n  = static_cast<int>(X.size());
    int nf = static_cast<int>(X[0].size());
    int nc = find_num_classes(y);
    if (nc < 2) return {};

    SplitResult best;
    for (int f = 0; f < nf; ++f) {
        std::vector<double> fv(n);
        for (int i = 0; i < n; ++i) fv[i] = X[i][f];
        auto bounds = equal_width_boundaries(fv, bin_count);
        int bc = bin_count;
        std::vector<std::vector<int>> hist(bc, std::vector<int>(nc, 0));
        for (int i = 0; i < n; ++i)
            hist[val_to_bin(fv[i], bounds)][y[i]]++;

        std::vector<int> lc(nc, 0), tot(nc, 0);
        int ln = 0, tn = 0;
        for (int b2 = 0; b2 < bc; ++b2)
            for (int c = 0; c < nc; ++c) { tot[c] += hist[b2][c]; tn += hist[b2][c]; }
        auto rc = tot; int rn = tn;

        for (int sb = 0; sb < bc - 1; ++sb) {
            for (int c = 0; c < nc; ++c) {
                lc[c] += hist[sb][c]; rc[c] -= hist[sb][c];
                ln    += hist[sb][c]; rn    -= hist[sb][c];
            }
            if (ln == 0 || rn == 0) continue;
            double wg = (double)ln/tn * gini_from_counts(lc)
                      + (double)rn/tn * gini_from_counts(rc);
            if (wg < best.best_gini) {
                best.best_gini      = wg;
                best.best_feature   = f;
                best.best_bin_index = sb;
                best.best_threshold = bounds[sb + 1];
                best.found          = true;
            }
        }
    }
    return best;
}

// =========================
// ══ YAMAN'S COMPONENTS ══
// =========================

// ----------------------------------------------------------------------------
// Y-1. GINI IMPURITY FOR A NODE
//      Computes the Gini impurity of the label distribution at one node.
//      Pure node → 0.0   Maximally mixed → close to 1.0
//      Used to report node quality and to verify splits reduce impurity.
// ----------------------------------------------------------------------------
double node_gini(const std::vector<int>& indices, const std::vector<int>& labels) {
    if (indices.empty()) return 0.0;
    std::map<int, int> counts;
    for (int idx : indices) counts[labels[idx]]++;
    int total = static_cast<int>(indices.size());
    double sum_sq = 0.0;
    for (const auto& [cls, cnt] : counts) {
        double p = static_cast<double>(cnt) / total;
        sum_sq += p * p;
    }
    return 1.0 - sum_sq;
}

// ----------------------------------------------------------------------------
// Y-2. MAJORITY CLASS
//      Returns the most frequent label among the samples at a node.
//      This becomes the node's predicted_class — used both for leaves
//      and as a fallback if traversal somehow reaches a non-leaf.
// ----------------------------------------------------------------------------
int majority_class(const std::vector<int>& indices, const std::vector<int>& labels) {
    std::map<int, int> counts;
    for (int idx : indices) counts[labels[idx]]++;
    return std::max_element(counts.begin(), counts.end(),
        [](const auto& a, const auto& b){ return a.second < b.second; })->first;
}

// ----------------------------------------------------------------------------
// Y-3. PURE-NODE CHECK
//      A node is pure if every sample in it shares the same class label.
//      Pure nodes must become leaves — splitting them further is pointless.
// ----------------------------------------------------------------------------
bool is_pure(const std::vector<int>& indices, const std::vector<int>& labels) {
    if (indices.empty()) return true;
    int first = labels[indices[0]];
    for (int idx : indices)
        if (labels[idx] != first) return false;
    return true;
}

// ----------------------------------------------------------------------------
// Y-4. BUILD DECISION TREE  ← Yaman's core contribution
//
//      Recursively grows the decision tree from a given node downwards.
//
//      Algorithm:
//        1. Record majority class at this node (always, even for internal nodes)
//        2. Check stopping conditions:
//              a. Node is pure → make leaf
//              b. Depth reached max_depth → make leaf
//              c. Too few samples to split → make leaf
//        3. Call Fatima's find_best_histogram_split() on the samples here
//        4. If no valid split found → make leaf
//        5. Partition sample indices: left (< threshold), right (> = threshold)
//        6. Call Faraz's split_node() to attach children
//        7. Recurse into left and right children
//
//      Parameters:
//        node              – current node (already has sample_indices set)
//        data              – full dataset (Faraz's Dataset struct)
//        max_depth         – maximum tree depth (prevents overfitting)
//        min_samples_split – don't split if node has fewer samples than this
//        bin_count         – number of histogram bins for Fatima's finder
// ----------------------------------------------------------------------------
void build_tree(
    Node*           node,
    const Dataset&  data,
    int             max_depth,
    int             min_samples_split,
    int             bin_count)
{
    // ── Step 1: Always record majority class ─────────────────────────────────
    const std::vector<int>& indices = get_samples_for_node(node);
    node->predicted_class = majority_class(indices, data.labels);

    // ── Step 2: Stopping conditions ──────────────────────────────────────────
    if (is_pure(indices, data.labels) ||
        node->depth >= max_depth      ||
        (int)indices.size() < min_samples_split)
    {
        node->is_leaf = true;
        return;
    }

    // ── Step 3: Build X / y sub-arrays for Fatima's split-finder ─────────────
    int nf = static_cast<int>(data.features[0].size());
    std::vector<std::vector<double>> X(indices.size(), std::vector<double>(nf));
    std::vector<int> y(indices.size());
    for (int i = 0; i < (int)indices.size(); ++i) {
        int s = indices[i];
        for (int f = 0; f < nf; ++f)
            X[i][f] = static_cast<double>(data.features[s][f]);
        y[i] = data.labels[s];
    }

    // ── Step 4: Find best split (Fatima's function) ───────────────────────────
    auto split_start = std::chrono::high_resolution_clock::now();
    SplitResult sr = find_best_histogram_split(X, y, bin_count);
    auto split_end = std::chrono::high_resolution_clock::now();
    g_split_eval_time_sec += std::chrono::duration<double>(split_end - split_start).count();

    if (!sr.found) {
        node->is_leaf = true;
        return;
    }

    // ── Step 5: Record split info in node ────────────────────────────────────
    node->feature_index = sr.best_feature;
    node->threshold     = static_cast<float>(sr.best_threshold);

    // ── Step 6: Partition sample indices ─────────────────────────────────────
    std::vector<int> left_idx, right_idx;
    for (int s : indices) {
        if (data.features[s][sr.best_feature] < node->threshold)
            left_idx.push_back(s);
        else
            right_idx.push_back(s);
    }

    // Guard: degenerate split → make leaf
    if (left_idx.empty() || right_idx.empty()) {
        node->is_leaf = true;
        return;
    }

    // ── Step 7: Create children (Faraz's split_node) and recurse ─────────────
    auto [left_child, right_child] = split_node(node, left_idx, right_idx);
    build_tree(left_child,  data, max_depth, min_samples_split, bin_count);
    build_tree(right_child, data, max_depth, min_samples_split, bin_count);
}

// -------------------------------------
// ZUHAA'S PREDICTION AND EVALUATION
// PREDICT — single sample
// -------------------------------------
int predict(const Node* node, const std::vector<float>& sample) {
    // Leaf or safety fallback
    if (node->is_leaf || node->left == nullptr || node->right == nullptr)
        return node->predicted_class;

    if (sample[node->feature_index] < node->threshold)
        return predict(node->left,  sample);
    else
        return predict(node->right, sample);
}

// -------------------------------------
// PREDICT BATCH — all test-set samples
// -------------------------------------
std::vector<int> predict_batch(
    const Node*             root,
    const Dataset&          data,
    const std::vector<int>& indices)
{
    std::vector<int> preds;
    preds.reserve(indices.size());
    for (int idx : indices)
        preds.push_back(predict(root, data.features[idx]));
    return preds;
}

// ---------
// ACCURACY
// ---------
double compute_accuracy(
    const std::vector<int>& predictions,
    const std::vector<int>& true_labels)
{
    if (predictions.size() != true_labels.size() || predictions.empty())
        throw std::invalid_argument("Size mismatch or empty vectors.");
    int correct = 0;
    for (int i = 0; i < (int)predictions.size(); ++i)
        if (predictions[i] == true_labels[i]) ++correct;
    return static_cast<double>(correct) / static_cast<double>(predictions.size());
}

// -------------
// TREE PRINTER
// -------------
void print_tree(
    const Node*             node,
    const Dataset&          data,
    const std::vector<std::string>& feat_names,
    const std::string&      prefix = "",
    bool                    is_left = true)
{
    if (!node) return;

    std::string connector = prefix.empty() ? "" : (is_left ? "├── " : "└── ");
    std::string new_prefix = prefix + (prefix.empty() ? "" : (is_left ? "│   " : "    "));

    int n = static_cast<int>(node->sample_indices.size());
    double g = node_gini(node->sample_indices, data.labels);

    if (node->is_leaf || (!node->left && !node->right)) {
        std::cout << prefix << connector
                  << "[LEAF] class=" << node->predicted_class
                  << "  samples=" << n
                  << "  gini=" << std::fixed << std::setprecision(4) << g
                  << "\n";
    } else {
        std::string fname = (node->feature_index >= 0 &&
                             node->feature_index < (int)feat_names.size())
                            ? feat_names[node->feature_index]
                            : "f" + std::to_string(node->feature_index);
        std::cout << prefix << connector
                  << "[depth " << node->depth << "]  "
                  << fname << " <= " << std::fixed << std::setprecision(4) << node->threshold
                  << "  |  samples=" << n
                  << "  gini=" << g
                  << "\n";
        print_tree(node->left,  data, feat_names, new_prefix, true);
        print_tree(node->right, data, feat_names, new_prefix, false);
    }
}

// ----------------
// TREE STATISTICS
// ----------------
struct TreeStats { int leaves = 0; int internals = 0; int max_depth = 0; };

void collect_stats(const Node* node, TreeStats& s) {
    if (!node) return;
    if (node->depth > s.max_depth) s.max_depth = node->depth;
    if (node->is_leaf || (!node->left && !node->right)) {
        ++s.leaves;
    } else {
        ++s.internals;
        collect_stats(node->left,  s);
        collect_stats(node->right, s);
    }
}

void print_tree_stats(const Node* root) {
    TreeStats s;
    collect_stats(root, s);
    std::cout << "\n[Tree Statistics]\n";
    std::cout << "  Leaf nodes    : " << s.leaves    << "\n";
    std::cout << "  Internal nodes: " << s.internals << "\n";
    std::cout << "  Max depth     : " << s.max_depth << "\n";
    std::cout << "  Total nodes   : " << (s.leaves + s.internals) << "\n";
}

// ----------------
// MEMORY CLEANUP
// ----------------
void delete_tree(Node* node) {
    if (!node) return;
    delete_tree(node->left);
    delete_tree(node->right);
    delete node;
}

// =======
// TESTS
// =======

// ── Test 1: Full pipeline — load, split, build, predict, accuracy ─────────
static void test_full_pipeline() {
    std::cout << "\n[Test 1: Full Pipeline]\n";

    Dataset data = load_csv("letter-recognition.csv", true);
    std::cout << "  Loaded " << data.features.size() << " samples, "
              << data.features[0].size() << " features, "
              << "feature names: ";
    for (const auto& n : data.feature_names) std::cout << n << " ";
    std::cout << "\n";

    auto [train_idx, test_idx] = train_test_split(data, 0.8f, 42);
    std::cout << "  Train: " << train_idx.size()
              << "  Test: "  << test_idx.size() << "\n";

    // Build tree (using Faraz's create_root_node)
    Node* root = create_root_node(train_idx);
    std::cout << "  Root node samples: " << get_samples_for_node(root).size() << "\n";
    std::cout << "  Root gini before split: "
              << std::fixed << std::setprecision(4)
              << node_gini(root->sample_indices, data.labels) << "\n";

    g_split_eval_time_sec = 0.0;   // reset before timing

    auto start = std::chrono::high_resolution_clock::now();
    build_tree(root, data, /*max_depth=*/5, /*min_samples_split=*/2, /*bin_count=*/8);
    auto end = std::chrono::high_resolution_clock::now();
    double train_time = std::chrono::duration<double>(end - start).count();

    double tree_building_time = train_time - g_split_eval_time_sec;

    std::cout << "\n[Timing Breakdown]\n";
    std::cout << "  Total training time   : " << train_time << " sec\n";
    std::cout << "  Split evaluation time : " << g_split_eval_time_sec
              << " sec  (" << (g_split_eval_time_sec / train_time * 100.0) << "%)\n";
    std::cout << "  Tree building time    : " << tree_building_time
              << " sec  (" << (tree_building_time / train_time * 100.0) << "%)\n";

    // Print tree structure
    std::cout << "\n[Tree Structure]\n";
    print_tree(root, data, data.feature_names);
    print_tree_stats(root);

    // Predict and evaluate
    auto p_start = std::chrono::high_resolution_clock::now();
    auto preds = predict_batch(root, data, test_idx);
    auto p_end = std::chrono::high_resolution_clock::now();
    double pred_time = std::chrono::duration<double>(p_end - p_start).count();
    std::cout << "Prediction Time: " << pred_time << " seconds\n";

    std::vector<int> true_labels;
    for (int i : test_idx) true_labels.push_back(data.labels[i]);

    double acc = compute_accuracy(preds, true_labels);
    std::cout << "\n  Predictions : ";
    for (int p : preds) std::cout << p << " ";
    std::cout << "\n  True labels : ";
    for (int t : true_labels) std::cout << t << " ";
    std::cout << "\n  Test Accuracy: " << acc * 100.0 << "%\n";
    std::cout << "\n[Final Metrics]\n";
    std::cout << "Accuracy      : " << acc * 100 << "%\n";
    std::cout << "Train Time    : " << train_time << " sec\n";
    std::cout << "Predict Time  : " << pred_time << " sec\n";

    assert(acc >= 0.0 && acc <= 1.0);
    std::cout << "  [PASS]\n";

    delete_tree(root);
}

// ── Test 2: Pure node → immediate leaf, no split ──────────────────────────
static void test_pure_node_is_leaf() {
    std::cout << "\n[Test 2: Pure Node Becomes Leaf]\n";

    Dataset data;
    for (int i = 0; i < 8; ++i) {
        data.features.push_back({static_cast<float>(i), 1.0f});
        data.labels.push_back(0); // all same class
    }
    std::vector<int> idx = {0,1,2,3,4,5,6,7};

    Node* root = create_root_node(idx);
    build_tree(root, data, 10, 2, 4);

    assert(root->is_leaf);
    assert(root->predicted_class == 0);
    assert(root->left  == nullptr);
    assert(root->right == nullptr);
    std::cout << "  is_leaf=true, predicted_class=0\n  [PASS]\n";

    delete_tree(root);
}

// ── Test 3: Max-depth constraint ──────────────────────────────────────────
static void test_max_depth_respected() {
    std::cout << "\n[Test 3: Max Depth Constraint]\n";

    Dataset data = load_csv("letter-recognition.csv", true);
    auto [train_idx, test_idx] = train_test_split(data, 0.8f, 1);

    for (int max_d : {1, 2, 3}) {
        Node* root = create_root_node(train_idx);
        build_tree(root, data, max_d, 1, 8);

        TreeStats s;
        collect_stats(root, s);
        std::cout << "  max_depth=" << max_d
                  << "  actual_max_depth=" << s.max_depth
                  << "  leaves=" << s.leaves << "\n";
        assert(s.max_depth <= max_d);
        delete_tree(root);
    }
    std::cout << "  [PASS]\n";
}

// ── Test 4: Perfect separation → 100% accuracy ────────────────────────────
static void test_perfect_split_accuracy() {
    std::cout << "\n[Test 4: Perfect Separation]\n";

    // class 0: feature[0] in [1.0 .. 4.0], class 1: [5.0 .. 9.0]
    // Hold out a clearly non-boundary sample to avoid depending on
    // equality-at-threshold routing conventions.
    Dataset data;
    for (int i = 1; i <= 9; ++i) {
        data.features.push_back({static_cast<float>(i), 0.0f});
        data.labels.push_back(i <= 4 ? 0 : 1);
    }
    // indices: 0(1.0,c0) 1(2.0,c0) 2(3.0,c0) 3(4.0,c0)
    //          4(5.0,c1) 5(6.0,c1) 6(7.0,c1) 7(8.0,c1) 8(9.0,c1)
    std::vector<int> train_idx = {0,1,2,3,5,6,7,8};
    std::vector<int> test_idx  = {4}; // 5.0 -> class 1, non-boundary check

    Node* root = create_root_node(train_idx);
    build_tree(root, data, 5, 1, 8);

    auto preds = predict_batch(root, data, test_idx);
    std::vector<int> true_labels;
    for (int i : test_idx) true_labels.push_back(data.labels[i]);

    double acc = compute_accuracy(preds, true_labels);
    std::cout << "  Accuracy: " << acc * 100.0 << "%\n";
    assert(acc == 1.0);
    std::cout << "  [PASS] 100% on perfectly separable data.\n";

    delete_tree(root);
}

// ── Test 5: Gini of pure vs mixed node ────────────────────────────────────
static void test_gini_values() {
    std::cout << "\n[Test 5: Gini Impurity Values]\n";

    Dataset data;
    // All class 0 → gini = 0.0
    for (int i = 0; i < 5; ++i) { data.features.push_back({1.0f}); data.labels.push_back(0); }
    // Equal class 0 & 1 → gini = 0.5
    for (int i = 0; i < 5; ++i) { data.features.push_back({2.0f}); data.labels.push_back(1); }

    std::vector<int> pure_idx  = {0,1,2,3,4};
    std::vector<int> mixed_idx = {0,1,2,3,4,5,6,7,8,9};

    double g_pure  = node_gini(pure_idx,  data.labels);
    double g_mixed = node_gini(mixed_idx, data.labels);

    std::cout << "  Pure node gini : " << g_pure  << "  (expected 0.0)\n";
    std::cout << "  Mixed node gini: " << g_mixed << "  (expected 0.5)\n";

    assert(std::fabs(g_pure  - 0.0) < 1e-6);
    assert(std::fabs(g_mixed - 0.5) < 1e-6);
    std::cout << "  [PASS]\n";
}

// ── Test 6: Train accuracy vs test accuracy (overfitting check) ────────────
static void test_train_vs_test() {
    std::cout << "\n[Test 6: Train vs Test Accuracy]\n";

    Dataset data = load_csv("letter-recognition.csv", true);
    auto [train_idx, test_idx] = train_test_split(data, 0.8f, 7);

    Node* root = create_root_node(train_idx);
    build_tree(root, data, 10, 1, 8);

    auto train_preds = predict_batch(root, data, train_idx);
    auto test_preds  = predict_batch(root, data, test_idx);

    std::vector<int> train_true, test_true;
    for (int i : train_idx) train_true.push_back(data.labels[i]);
    for (int i : test_idx)  test_true.push_back(data.labels[i]);

    double train_acc = compute_accuracy(train_preds, train_true);
    double test_acc  = compute_accuracy(test_preds,  test_true);

    std::cout << "  Train accuracy : " << train_acc * 100.0 << "%\n";
    std::cout << "  Test  accuracy : " << test_acc  * 100.0 << "%\n";
    std::cout << "  (Train >= Test is expected — shows tree is fitting)\n";

    assert(train_acc >= test_acc - 0.01); // allow tiny float tolerance
    std::cout << "  [PASS]\n";

    delete_tree(root);
}

// MAIN
int main() {
    std::cout << "=============================================\n";
    std::cout << "   MILESTONE 1 — Decision Tree Builder\n";
    std::cout << "=============================================\n";

    try {
        test_full_pipeline();
        test_pure_node_is_leaf();
        test_max_depth_respected();
        test_perfect_split_accuracy();
        test_gini_values();
        test_train_vs_test();

        std::cout << "\n[All Tests Passed]\n";
    } catch (const std::exception& ex) {
        std::cerr << "Error: " << ex.what() << "\n";
        return 1;
    }
    return 0;
}

Overwriting decision_tree.cpp


In [98]:
!g++ tree.cpp -o tree
!./tree

   MILESTONE 1 — Decision Tree Builder

[Test 1: Full Pipeline]
  Loaded 14500 samples, 9 features, feature names: A1 A2 A3 A4 A5 A6 A7 A8 A9 
  Train: 11600  Test: 2900
  Root node samples: 11600
  Root gini before split: 0.3537

[Timing Breakdown]
  Total training time   : 0.1866 sec
  Split evaluation time : 0.1343 sec  (71.9442%)
  Tree building time    : 0.0524 sec  (28.0558%)

[Tree Structure]
[depth 0]  A7 <= 27.5000  |  samples=11600  gini=0.3537
[depth 1]  A7 <= 10.1250  |  samples=2004  gini=0.6441
[depth 2]  A7 <= 5.6250  |  samples=664  gini=0.0090
[depth 3]  A1 <= 47.7500  |  samples=662  gini=0.0030
[LEAF] class=1  samples=1  gini=0.0000
[LEAF] class=5  samples=661  gini=0.0000
[depth 3]  A1 <= 79.1250  |  samples=2  gini=0.5000
[LEAF] class=6  samples=1  gini=0.0000
[LEAF] class=3  samples=1  gini=0.0000
[depth 2]  A8 <= 34.1250  |  samples=1340  gini=0.4477
[depth 3]  A8 <= 30.7500  |  samples=456  gini=0.1305
[LEAF] class=1  samples=424  gini=0.0000
[LEAF] class=4  sam

In [96]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import pandas as pd
import time

data = pd.read_csv("letter-recognition.csv")

X = data.iloc[:, :-1]
y = data.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = DecisionTreeClassifier()

start = time.time()
model.fit(X_train, y_train)
end = time.time()

print("Sklearn Train Time:", end - start)
print("Sklearn Accuracy:", model.score(X_test, y_test))

Sklearn Train Time: 0.31217145919799805
Sklearn Accuracy: 0.9993266955031421
